# THIS NOTEBOOK HAS BEEN TESTED ONLY UNDER PYTHON 3.12
## It is advisable to use a venv to test this code

In [ ]:
import sys
import os

def encontrar_carpeta(nombre_subruta, niveles_max=5):
    """
    Busca la carpeta 'nombre_subruta' empezando en la carpeta actual
    y subiendo hasta 'niveles_max' niveles hacia arriba. Devuelve la
    ruta absoluta si la encuentra; si no, lanza un error claro.
    Independiente del sistema operativo y de dónde arrancó el notebook.
    """
    base = os.getcwd()
    for _ in range(niveles_max + 1):
        candidata = os.path.join(base, *nombre_subruta)
        if os.path.isdir(candidata):
            return candidata
        base = os.path.dirname(base)  # sube un nivel
    raise FileNotFoundError(
        f"No encontré la carpeta {os.path.join(*nombre_subruta)} "
        f"partiendo de {os.getcwd()}"
    )

# Buscar la carpeta de los módulos sin asumir dónde arrancó el notebook
carpeta_preprocessing = encontrar_carpeta(("scripts", "preprocessing"))

# Agregarla a la ruta de búsqueda de Python
sys.path.insert(0, carpeta_preprocessing)

print("Carpeta encontrada:", carpeta_preprocessing)


In [ ]:
## Descomentar si falla torch o numpy por version
#!pip uninstall torch torchvision torchaudio -y
#!pip cache purge

ruta_requirements = os.path.join(carpeta_preprocessing, "requirements.txt")

!{sys.executable} -m pip install -q -r "{ruta_requirements}"

In [ ]:
from pdf_preprocessor import process_pdf
from pbf_preprocessor import process_pbf
from json_preprocessor import procesar_jsonl
from csv_preprocessor import process_csv_file
import tqdm, os

In [ ]:
from tqdm import tqdm

carpeta_files = encontrar_carpeta(("src", "CORPUS CODEFEST AD ASTRA 2026"))

folders = [ls for ls in os.listdir(carpeta_files) if os.path.isdir(os.path.join(carpeta_files, ls))]

archivos_por_carpeta = {}

# Una barra de progreso por cada carpeta principal
for folder in folders:
    ruta_folder = os.path.join(carpeta_files, folder)

    # Primero recolectamos todos los archivos de esta carpeta (para conocer el total)
    todos = [
        os.path.join(root, file)
        for root, _, files in os.walk(ruta_folder)
        for file in files
    ]

    # Barra de progreso propia de esta carpeta, filtrando .DS_Store
    archivos = [
        f for f in tqdm(todos, desc=folder, unit="archivo")
        if os.path.basename(f) != ".DS_Store"
    ]

    archivos_por_carpeta[folder] = archivos

# Resumen final
print("\nResumen:")
for folder, archivos in archivos_por_carpeta.items():
    print(f"{folder}: {len(archivos)} archivos")

total = sum(len(archivos) for archivos in archivos_por_carpeta.values())
print(f"\nTotal: {total} archivos")

# Celda de ejecucion de todas las clases

In [ ]:
# Cada procesador se adapta a una firma común: (file, fenomeno) -> chunks
PROCESADORES = {
    ".pdf":  lambda file, fenomeno: process_pdf(pdf_path=file, fenomeno=fenomeno),
    ".pbf":  lambda file, fenomeno: process_pbf(pbf_path=file, fenomeno=fenomeno),
    ".json": lambda file, fenomeno: procesar_jsonl(path=file, fenomeno=fenomeno),
    ".csv":  lambda file, fenomeno: process_csv_file(csv_path=file, fenomeno=fenomeno),
}


def segmentar_archivos(modo_prueba: bool = False) -> str:
    todos_los_chunks = []
    formatos_procesados = set()

    for key in archivos_por_carpeta:
        fenomeno = key[1]
        files = archivos_por_carpeta[key]

        print(f"Procesando archivos para el fenómeno: {fenomeno}")

        for file in files:
            extension = os.path.splitext(file)[1].lower()

            # Ignorar formatos sin procesador
            procesador = PROCESADORES.get(extension)
            if procesador is None:
                continue

            # En modo prueba, procesar solo un archivo por formato
            if modo_prueba and extension in formatos_procesados:
                continue

            print(f"Procesando archivo: {os.path.basename(file)} para el fenómeno: {fenomeno}")

            chunks = procesador(file, fenomeno)

            todos_los_chunks.append(chunks)
            formatos_procesados.add(extension)

            # En modo prueba, terminar al reunir los formatos
            if modo_prueba and len(formatos_procesados) == len(PROCESADORES):
                break

        if modo_prueba and len(formatos_procesados) == len(PROCESADORES):
            break

    chunks_final = "\n".join(todos_los_chunks)

    if modo_prueba:
        print("\nPrueba terminada.")
        print(f"Formatos procesados: {formatos_procesados}")

    with open("metadata.jsonl", "w", encoding="utf-8") as f:
        f.write(chunks_final)

    return chunks_final

In [ ]:
import json

# chunks puede venir como string JSONL (una linea JSON por chunk) o ya como lista de dicts
if isinstance(chunks, str):
    registros = [json.loads(linea) for linea in chunks.splitlines() if linea.strip()]
else:
    registros = chunks

print(json.dumps(registros, indent=2, ensure_ascii=False))

In [ ]:
#pip install mapbox-vector-tile

In [ ]:
"""import mapbox_vector_tile

with open(r"C:\git local\codefest\base_de_conocimiento\src\CORPUS CODEFEST AD ASTRA 2026\F3_Dinamicas_Territoriales\Amazon_Underworld\tiles\3\2\AMAZONUW_3.pbf", "rb") as f:
    data = f.read()

tile = mapbox_vector_tile.decode(data)

# Ver las capas disponibles
for layer_name, layer in tile.items():
    print(f"Capa: {layer_name} -> {len(layer['features'])} features")

# Inspeccionar features de una capa
for layer_name, layer in tile.items():
    for feature in layer["features"][:5]:
        print(feature["geometry"], feature["properties"])"""

In [ ]:
#pip install --user -q mapbox-vector-tile shapely matplotlib

In [ ]:
"""import gzip
import mapbox_vector_tile
import matplotlib.pyplot as plt
from shapely.geometry import shape

ruta = r"C:\git local\codefest\base_de_conocimiento\src\CORPUS CODEFEST AD ASTRA 2026\F3_Dinamicas_Territoriales\Amazon_Underworld\tiles\4\4\AMAZONUW_8.pbf"

with open(ruta, "rb") as f:
    data = f.read()

# Descomprime si viene en gzip
if data[:2] == b"\x1f\x8b":
    data = gzip.decompress(data)

tile = mapbox_vector_tile.decode(data)

fig, ax = plt.subplots(figsize=(10, 10))

for layer_name, layer in tile.items():
    for feature in layer["features"]:
        geom = shape(feature["geometry"])
        gtype = geom.geom_type

        if gtype == "Point":
            ax.plot(geom.x, geom.y, "o", markersize=2)
        elif gtype == "MultiPoint":
            for p in geom.geoms:
                ax.plot(p.x, p.y, "o", markersize=2)
        elif gtype == "LineString":
            x, y = geom.xy
            ax.plot(x, y, linewidth=0.5)
        elif gtype == "MultiLineString":
            for line in geom.geoms:
                x, y = line.xy
                ax.plot(x, y, linewidth=0.5)
        elif gtype == "Polygon":
            x, y = geom.exterior.xy
            ax.fill(x, y, alpha=0.4)
        elif gtype == "MultiPolygon":
            for poly in geom.geoms:
                x, y = poly.exterior.xy
                ax.fill(x, y, alpha=0.4)

ax.set_aspect("equal")
ax.axis("off")
plt.savefig("tile.png", dpi=150, bbox_inches="tight")
plt.show()

print("Imagen guardada como tile.png")"""

In [ ]:
#chunks=process_pbf(fenomeno=3, fuente="AMAZONUW_3.pbf", pbf_path=r"C:\git local\codefest\base_de_conocimiento\src\CORPUS CODEFEST AD ASTRA 2026\F3_Dinamicas_Territoriales\Amazon_Underworld\tiles\3\2\AMAZONUW_3.pbf")

In [ ]:
"""import json

# chunks puede venir como string JSONL (una linea JSON por chunk) o ya como lista de dicts
if isinstance(chunks, str):
    registros = [json.loads(linea) for linea in chunks.splitlines() if linea.strip()]
else:
    registros = chunks

print(json.dumps(registros, indent=2, ensure_ascii=False))"""